In [1]:
import pandas as pd

# Notebook lives in /notebooks, so the data is one level up
src = "../data/processed/london_enriched_2026-09-22.csv"
out = "../data/processed/borough_risk_summary.csv"

df = pd.read_csv(src, low_memory=False)

# Keep only genuinely inspected businesses (ratings 0 to 5)
df["rating"] = pd.to_numeric(df["RatingValue"], errors="coerce")
df = df[df["rating"].between(0, 5)]

# Failure = rating 2 or below
df["fail"] = df["rating"] <= 2

summary = (
    df.groupby(["LocalAuthorityName", "BusinessType"])
      .agg(businesses=("fail", "size"), fail_rate=("fail", "mean"))
      .reset_index()
      .rename(columns={"LocalAuthorityName": "borough",
                       "BusinessType": "business_type"})
)

# Drop small groups so tiny counts don't create misleading rates
summary = summary[summary["businesses"] >= 20]
summary["fail_rate"] = summary["fail_rate"].round(3)
summary = summary.sort_values(["borough", "fail_rate"], ascending=[True, False])

summary.to_csv(out, index=False)

print(f"Rated businesses used: {len(df):,}")
print(f"Overall London fail rate: {df['fail'].mean():.1%}")
print(f"Boroughs: {summary['borough'].nunique()}")
print(f"Rows in summary: {len(summary)}")
summary.head(10)

Rated businesses used: 71,390
Overall London fail rate: 5.6%
Boroughs: 33
Rows in summary: 326


,borough,business_type,businesses,fail_rate
8,Barking and Dagenham,Retailers - other,390,0.128
11,Barking and Dagenham,Takeaway/sandwich shop,216,0.111
7,Barking and Dagenham,Restaurant/Cafe/Canteen,210,0.086
6,Barking and Dagenham,Pub/bar/nightclub,24,0.083
0,Barking and Dagenham,Caring Premises,60,0.050
9,Barking and Dagenham,Retailers - supermarkets/hypermarkets,39,0.026
5,Barking and Dagenham,Other catering premises,142,0.021
4,Barking and Dagenham,Mobile caterer,47,0.000
10,Barking and Dagenham,School/college/university,92,0.000
21,Barnet,Retailers - other,460,0.120
